##### Import Libaries and Functions

In [0]:
# import boto3
import os

from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col
from pyspark.sql import DataFrame

import delta_table_creator, transform
from delta import DeltaTable  # type: ignore

In [0]:
# Import the module
import sys
sys.path.append("../aws_utils")

import aws_utils
# Get Environment Prefix

env = aws_utils.get_env_prefix()

##### Create widgets with default values. Change them if needed from UI
* run_date : '2024-02-28'
* unity : True | False

In case of Unity Catalog, the appropriate cluster should be attached

##### Get the values of widgets and initialize the environment variable
Ensure that widgets are assigned the wanted values before proceeding with the next step.

In [0]:
# Takes the values d for dev, qa for test and p for prod depending in which workspace the init variables script is running
#env = os.environ["S3PREFIX"] 
# Takes the values of widgets

result = spark.sql( f'''
    SELECT max(BusinessDate) max_date
        FROM `{env}-cdh`.primary.AuctionRealEstate
        '''
)
max_date = result.collect()[0]['max_date']
run_date= max_date.strftime('%Y-%m-%d')
unity = 'True'

# run_date='2024-01-12'
# unity = 'True'

table_name = 'DAuctionRealEstate'

# """
# Takes the s3 parameter value from ssm

# ssm_client = boto3.client("ssm", region_name="eu-central-1") # Initialize the SSM client

# response = ssm_client.get_parameter(
#     Name='/cfn/a360/matching/conf_s3_path',
#     WithDecryption=False
# )

# s3_path = response['Parameter']['Value']
# """

##### Initialize the schema

In [0]:
schema = (
    "AuctionRealEstateId string"
    ",OriginSourceSystem string"
    ",SourceSystem string"
    ",IngestionTimestamp timestamp"
    ",BatchId string"
    ",AuctionRealEstateKey string"
    ",AuctionRealEstateCode string"
    ",AuctionCode string"
    ",AuctionKey string"
    ",AuctionObjKey string"
    ",RealEstateGroupCode int"
    ",DebtorPartyKey string"
    ",OriginDebtorTaxRegistrationNumber string"
    ",DebtorTaxRegistrationNumber string"
    ",OriginDebtorDescription string"
    ",DebtorDescription string"
    ",DebtorReferenceInfo string"
    ",AuctionDate date"
    ",AuctionPublishedDate date"
    ",AuctionPostedDate date"
    ",PartyKey string"
    ",RealEstateKey string"
    ",RealEstateOwnershipKey string"
    ",CaseCode int"
    ",AuctionFirstOfferPrice string"
    ",SeizureReportNumber string"
    ",SeizureReportYear int"
    ",HastenerCode string"
    ",Hastener string"
    ",HastenerTaxRegistrationNumber string"
    ",AuctionFinalPrice string"
    ",AuctionStatusCode string"
    ",AuctionStatus string"
    ",AuctioneerName string"
    ",AuctioneerEmail string"
    ",AuctioneerPhone string"
    ",TitleGR string"
    ",Area string"
    ",District string"
    ",Municipality string"
    ",Prefecture string"
    ",GeoCoordinatesX string"
    ",GeoCoordinatesY string"
    ",GeoLevel string"
    ",Address string"
    ",PostalCode int"
    ",Category string"
    ",SubCategory string"
    ",LandeaUrl string"
    ",FloorRawValue string"
    ",ConstructionYear int"
    ",SurfaceSqm float"
    ",Tags string"
    ",TagValues string"
    ",ActiveFlag boolean"
    ",TNMergeCode string"
    ",LandRegistration string"
    ",BidderCode string"
    ",DateEntered string"
    ",EnteredBy string"
    ",AuctionOwnershipTypeCode int"
    ",AuctionOwnershipType string"
    ",FirstBidder string"
    ",LawFirm string"
    ",FileDate date"
    ",FileName string"
    ",BusinessDate date"
    ",rcg_provider_auctionobjid int"
    ",rcg_provider_mbp_price float"
    ",rcg_provider_uw_price float"
    ",rcg_provider_acquired_by string"
    ",rcg_provider_file_date date"
    ",rcg_provider_eligibility string"
    ",rcg_provider_initiator string"
    ",rcg_provider_Servicer string"
    ",rcg_provider_Auction_result_updated string"
    ",rcg_provider_suspension_reason_grouped string"
    ",rcg_provider_suspension_reason_full string"
    ",rcg_provider_eligibility_flag_property boolean"
    ",rcg_provider_eligible_for_reschedule boolean"
    ",rcg_provider_marketability_score decimal(5,2)"
    ",rcg_provider_auction_round int"
)


##### Create - Delete Delta Table

In [0]:
partition_columns = ["BusinessDate"]
derivative_table_path = ""
 
#================== Create Delta Table ==================
if unity == "True":
    derivative_table_path = f"`{env}-cdh`.derivativeactive.{table_name}"
    delta_table_creator.create_unity_delta_table(
            spark, derivative_table_path, partition_columns, schema
        )
else:
    derivative_table_path = f"s3://{env}-ec1-cepal-derivative-zone/{table_name}"
    if not DeltaTable.isDeltaTable(spark, derivative_table_path):
        delta_table_creator.create_delta_table(
            spark, derivative_table_path, partition_columns, schema
        )
 
 
#================== Delete From Delta Table ==================
 
transform.delete_from_unity_table(
    spark,
    derivative_table_path,
    run_date,  # type: ignore
)

##### Create wanted Delta Tables with selected fields & Temporary Views for each delta table

##### AuctionRealEstate

In [0]:
auctionrealestate = spark.sql(
f'''
SELECT 
    AuctionRealEstateId 
    ,OriginSourceSystem
    ,SourceSystem 
    ,IngestionTimestamp 
    ,BatchId 
    ,AuctionRealEstateKey 
    ,AuctionRealEstateCode 
    ,AuctionCode 
    ,AuctionObjKey 
    ,RealEstateGroupCode 
    ,DebtorPartyKey 
    ,OriginDebtorTaxRegistrationNumber 
    ,OriginDebtorDescription 
    ,AuctionDate 
    ,AuctionPublishedDate 
    ,AuctionPostedDate 
    ,PartyKey 
    ,RealEstateKey 
    ,RealEstateOwnershipKey 
    ,CaseCode 
    ,AuctionFirstOfferPrice 
    ,SeizureReportNumber 
    ,SeizureReportYear 
    ,HastenerCode 
    ,Hastener 
    ,HastenerTaxRegistrationNumber 
    ,AuctionFinalPrice 
    ,AuctionStatusCode 
    ,AuctionStatus 
    ,AuctioneerName 
    ,AuctioneerEmail 
    ,AuctioneerPhone 
    ,TitleGR 
    ,Area 
    ,District 
    ,Municipality 
    ,Prefecture 
    ,GeoCoordinatesX 
    ,GeoCoordinatesY 
    ,GeoLevel 
    ,Address 
    ,PostalCode 
    ,Category 
    ,SubCategory 
    ,LandeaUrl 
    ,FloorRawValue 
    ,ConstructionYear 
    ,SurfaceSqm 
    ,Tags 
    ,TagValues 
    ,ActiveFlag 
    ,TNMergeCode 
    ,LandRegistration 
    ,BidderCode 
    ,DateEntered 
    ,EnteredBy 
    ,AuctionOwnershipTypeCode 
    ,AuctionOwnershipType 
    ,FirstBidder 
    ,LawFirm 
    ,FileDate 
    ,FileName 
    ,BusinessDate
    ,rcg_provider_auctionobjid
    ,rcg_provider_mbp_price
    ,rcg_provider_uw_price
    ,rcg_provider_acquired_by
    ,rcg_provider_file_date
    ,rcg_provider_eligibility
    ,rcg_provider_initiator
    ,rcg_provider_Servicer
    ,rcg_provider_Auction_result_updated
    ,rcg_provider_suspension_reason_grouped
    ,rcg_provider_suspension_reason_full
    ,rcg_provider_eligibility_flag_property
    ,rcg_provider_eligible_for_reschedule
    ,rcg_provider_marketability_score
    ,rcg_provider_auction_round
FROM  `{env}-cdh`.primary.auctionrealestate
WHERE BusinessDate = {run_date!r} and SourceSystem='extfile' 
'''
)

auctionrealestate.createOrReplaceTempView("auctionrealestate")

##### Create Temporary Views for Calculated fields

In [0]:
auctionrealestate_expanded = f"""
WITH split_auctionrealestate AS (
    SELECT
        *,
        CASE 
            WHEN OriginDebtorTaxRegistrationNumber LIKE '%||%' 
                THEN SPLIT(REPLACE(OriginDebtorTaxRegistrationNumber, '||', '###'), '###')
            ELSE ARRAY(OriginDebtorTaxRegistrationNumber)
        END AS splited_DebtorVatNumber,

        CASE 
            WHEN OriginDebtorDescription LIKE '%||%' 
                THEN SPLIT(REPLACE(OriginDebtorDescription, '||', '###'), '###')
            ELSE ARRAY(OriginDebtorDescription)
        END AS splited_DebtorDescr
    FROM 
        auctionrealestate
),
array_truncated AS (
    SELECT
        *,
        -- Find the common minimum length
        size(splited_DebtorVatNumber) AS vat_size,
        size(splited_DebtorDescr) AS descr_size,
        array_min(array(size(splited_DebtorVatNumber), size(splited_DebtorDescr))) AS common_size
    FROM split_auctionrealestate
),
aligned_array AS (
    SELECT
        *,
        SLICE(splited_DebtorVatNumber, 1, common_size) AS aligned_vat,
        SLICE(splited_DebtorDescr, 1, common_size) AS aligned_descr
    FROM array_truncated
),
exploded_auctionrealestate AS (
    SELECT
        ar.*,
        pos,
        DebtorTaxRegistrationNumber,
        get(aligned_descr, pos) AS DebtorDescription
    FROM aligned_array ar
    LATERAL VIEW POSEXPLODE(aligned_vat) AS pos, DebtorTaxRegistrationNumber
)
SELECT
    *,
    SHA2(
        CONCAT_WS(';', AuctionRealEstateCode, AuctionObjKey, OriginSourceSystem, TRIM(DebtorTaxRegistrationNumber), 'extfile'), 
        256
    ) AS AuctionRealEstateIdExpanded,
    SPLIT(AuctionRealEstateCode, '-')[0] AS UniqueCodeSplit,
    CONCAT_WS(';', AuctionRealEstateCode, AuctionObjKey, OriginSourceSystem, TRIM(DebtorTaxRegistrationNumber)) AS AuctionRealEstateKeyExpanded
FROM exploded_auctionrealestate

"""

# Execute the SQL query
auctionrealestate_expanded = spark.sql(auctionrealestate_expanded)
auctionrealestate_expanded.createOrReplaceTempView("auctionrealestate_expanded")

In [0]:
from pyspark.sql.functions import col, sha2, concat_ws, split, trim, lit, regexp_replace, regexp_extract

# Ensure the column is treated as a string
auctionrealestate_expanded = auctionrealestate_expanded.withColumn(
    "DebtorTaxRegistrationNumber",
    col("DebtorTaxRegistrationNumber").cast("string")
)


# Add the existing columns and new columns to the DataFrame
auctionrealestate_expanded_cleaned = auctionrealestate_expanded.withColumn(
    "DebtorTaxRegistrationNumberCleaned",
    when( # when 9 consecutive digits, no less and no more
        (col("DebtorTaxRegistrationNumber").rlike(r'\d{9}')) &  
        (~col("DebtorTaxRegistrationNumber").rlike(r'\d{10,}')),
        regexp_extract(col("DebtorTaxRegistrationNumber"), r'\d{9}', 0)
    ).when( #9 when null
        col("DebtorTaxRegistrationNumber").isNull(),  # Column is empty
        ""
    ).otherwise("Invalid")
).withColumn(
    "DebtorReferenceInfo",
    when( # when 9 consecutive digits, no less and no more
        (col("DebtorTaxRegistrationNumber").rlike(r'\d{9}')) &  
        (~col("DebtorTaxRegistrationNumber").rlike(r'\d{10,}')),
        regexp_replace(
                col("DebtorTaxRegistrationNumber"), 
                regexp_extract(col("DebtorTaxRegistrationNumber"), r'[0-9]{9}', 0), '')
    ).when( #9 when null
        col("DebtorTaxRegistrationNumber").isNull(),  # Column is empty
        ""
    ).otherwise(col("DebtorTaxRegistrationNumber"))
)


auctionrealestate_expanded_cleaned.createOrReplaceTempView("auctionrealestate_expanded_cleaned")

In [0]:
auctionrealestate_extfile = spark.sql(
f'''
SELECT 
    AuctionRealEstateIdExpanded as AuctionRealEstateId 
    ,OriginSourceSystem 
    ,SourceSystem 
    ,IngestionTimestamp 
    ,BatchId 
    ,AuctionRealEstateKeyExpanded as AuctionRealEstateKey
    ,AuctionRealEstateCode 
    ,AuctionCode 
    ,UniqueCodeSplit as AuctionKey 
    ,AuctionObjKey 
    ,RealEstateGroupCode 
    ,DebtorPartyKey 
    ,OriginDebtorTaxRegistrationNumber 
    ,DebtorTaxRegistrationNumberCleaned as DebtorTaxRegistrationNumber 
    ,OriginDebtorDescription 
    ,DebtorDescription 
    ,DebtorReferenceInfo
    ,AuctionDate 
    ,AuctionPublishedDate 
    ,AuctionPostedDate 
    ,PartyKey 
    ,RealEstateKey 
    ,RealEstateOwnershipKey 
    ,CaseCode 
    ,AuctionFirstOfferPrice 
    ,SeizureReportNumber 
    ,SeizureReportYear 
    ,HastenerCode 
    ,Hastener 
    ,HastenerTaxRegistrationNumber 
    ,AuctionFinalPrice 
    ,AuctionStatusCode 
    ,AuctionStatus 
    ,AuctioneerName 
    ,AuctioneerEmail 
    ,AuctioneerPhone 
    ,TitleGR 
    ,Area 
    ,District 
    ,Municipality 
    ,Prefecture 
    ,GeoCoordinatesX 
    ,GeoCoordinatesY 
    ,GeoLevel 
    ,Address 
    ,PostalCode 
    ,Category 
    ,SubCategory 
    ,LandeaUrl 
    ,FloorRawValue 
    ,ConstructionYear 
    ,SurfaceSqm 
    ,Tags 
    ,TagValues 
    ,ActiveFlag 
    ,TNMergeCode 
    ,LandRegistration 
    ,BidderCode 
    ,DateEntered 
    ,EnteredBy 
    ,AuctionOwnershipTypeCode 
    ,AuctionOwnershipType 
    ,FirstBidder 
    ,LawFirm 
    ,FileDate 
    ,FileName 
    ,BusinessDate 
    ,rcg_provider_auctionobjid
    ,rcg_provider_mbp_price
    ,rcg_provider_uw_price
    ,rcg_provider_acquired_by
    ,rcg_provider_file_date
    ,rcg_provider_eligibility
    ,rcg_provider_initiator
    ,rcg_provider_Servicer
    ,rcg_provider_Auction_result_updated
    ,rcg_provider_suspension_reason_grouped
    ,rcg_provider_suspension_reason_full
    ,rcg_provider_eligibility_flag_property
    ,rcg_provider_eligible_for_reschedule
    ,rcg_provider_marketability_score
    ,rcg_provider_auction_round
FROM  auctionrealestate_expanded_cleaned
'''
)

auctionrealestate_extfile.createOrReplaceTempView("auctionrealestate_extfile")

##### auctionrealestate_thesis

In [0]:
auctionrealestate_thesis = spark.sql(
f'''
SELECT 
    AuctionRealEstateId 
    ,OriginSourceSystem 
    ,SourceSystem 
    ,IngestionTimestamp 
    ,BatchId 
    ,AuctionRealEstateKey 
    ,AuctionRealEstateCode 
    ,AuctionCode 
    ,NULL as AuctionKey 
    ,AuctionObjKey 
    ,RealEstateGroupCode 
    ,DebtorPartyKey 
    ,OriginDebtorTaxRegistrationNumber 
    ,OriginDebtorTaxRegistrationNumber AS DebtorTaxRegistrationNumber 
    ,OriginDebtorDescription 
    ,OriginDebtorDescription AS DebtorDescription
    ,NULL AS DebtorReferenceInfo
    ,AuctionDate 
    ,AuctionPublishedDate 
    ,AuctionPostedDate 
    ,PartyKey 
    ,RealEstateKey 
    ,RealEstateOwnershipKey 
    ,CaseCode 
    ,AuctionFirstOfferPrice 
    ,SeizureReportNumber 
    ,SeizureReportYear 
    ,HastenerCode 
    ,Hastener 
    ,HastenerTaxRegistrationNumber 
    ,AuctionFinalPrice 
    ,AuctionStatusCode 
    ,AuctionStatus 
    ,AuctioneerName 
    ,AuctioneerEmail 
    ,AuctioneerPhone 
    ,TitleGR 
    ,Area 
    ,District 
    ,Municipality 
    ,Prefecture 
    ,GeoCoordinatesX 
    ,GeoCoordinatesY 
    ,GeoLevel 
    ,Address 
    ,PostalCode 
    ,Category 
    ,SubCategory 
    ,LandeaUrl 
    ,FloorRawValue 
    ,ConstructionYear 
    ,SurfaceSqm 
    ,Tags 
    ,TagValues 
    ,ActiveFlag 
    ,TNMergeCode 
    ,LandRegistration 
    ,BidderCode 
    ,DateEntered 
    ,EnteredBy 
    ,AuctionOwnershipTypeCode 
    ,AuctionOwnershipType 
    ,FirstBidder 
    ,LawFirm 
    ,FileDate 
    ,FileName 
    ,BusinessDate
    ,CAST(NULL AS INT) AS rcg_provider_auctionobjid
    ,CAST(NULL AS FLOAT) AS rcg_provider_mbp_price
    ,CAST(NULL AS FLOAT) AS rcg_provider_uw_price
    ,CAST(NULL AS STRING) AS rcg_provider_acquired_by
    ,CAST(NULL AS DATE) AS rcg_provider_file_date
    ,CAST(NULL AS STRING) AS rcg_provider_eligibility
    ,CAST(NULL AS STRING) AS rcg_provider_initiator
    ,CAST(NULL AS STRING) AS rcg_provider_Servicer
    ,CAST(NULL AS STRING) AS rcg_provider_Auction_result_updated
    ,CAST(NULL AS STRING) AS rcg_provider_suspension_reason_grouped
    ,CAST(NULL AS STRING) AS rcg_provider_suspension_reason_full
    ,CAST(NULL AS BOOLEAN) AS rcg_provider_eligibility_flag_property
    ,CAST(NULL AS BOOLEAN) AS rcg_provider_eligible_for_reschedule
    ,CAST(NULL AS DECIMAL(5,2)) AS rcg_provider_marketability_score
    ,CAST(NULL AS INT) AS rcg_provider_auction_round

FROM  `{env}-cdh`.primary.auctionrealestate
WHERE BusinessDate = {run_date!r} and SourceSystem='thesis'
'''
)

auctionrealestate_thesis.createOrReplaceTempView("auctionrealestate_thesis")

##### acutionrealestate_qca

In [0]:
auctionrealestate_qca = spark.sql(
f'''
SELECT 
    AuctionRealEstateId 
    ,OriginSourceSystem 
    ,SourceSystem 
    ,IngestionTimestamp 
    ,BatchId 
    ,AuctionRealEstateKey 
    ,AuctionRealEstateCode 
    ,AuctionCode 
    ,NULL as AuctionKey 
    ,AuctionObjKey 
    ,RealEstateGroupCode 
    ,DebtorPartyKey 
    ,OriginDebtorTaxRegistrationNumber 
    ,NULL AS DebtorTaxRegistrationNumber 
    ,OriginDebtorDescription 
    ,NULL AS DebtorDescription
    ,NULL AS DebtorReferenceInfo
    ,AuctionDate 
    ,AuctionPublishedDate 
    ,AuctionPostedDate 
    ,PartyKey 
    ,RealEstateKey 
    ,RealEstateOwnershipKey 
    ,CaseCode 
    ,AuctionFirstOfferPrice 
    ,SeizureReportNumber 
    ,SeizureReportYear 
    ,HastenerCode 
    ,Hastener 
    ,HastenerTaxRegistrationNumber 
    ,AuctionFinalPrice 
    ,AuctionStatusCode 
    ,AuctionStatus 
    ,AuctioneerName 
    ,AuctioneerEmail 
    ,AuctioneerPhone 
    ,TitleGR 
    ,Area 
    ,District 
    ,Municipality 
    ,Prefecture 
    ,GeoCoordinatesX 
    ,GeoCoordinatesY 
    ,GeoLevel 
    ,Address 
    ,PostalCode 
    ,Category 
    ,SubCategory 
    ,LandeaUrl 
    ,FloorRawValue 
    ,ConstructionYear 
    ,SurfaceSqm 
    ,Tags 
    ,TagValues 
    ,ActiveFlag 
    ,TNMergeCode 
    ,LandRegistration 
    ,BidderCode 
    ,DateEntered 
    ,EnteredBy 
    ,AuctionOwnershipTypeCode 
    ,AuctionOwnershipType 
    ,FirstBidder 
    ,LawFirm 
    ,FileDate 
    ,FileName 
    ,BusinessDate
    ,CAST(NULL AS INT) AS rcg_provider_auctionobjid
    ,CAST(NULL AS FLOAT) AS rcg_provider_mbp_price
    ,CAST(NULL AS FLOAT) AS rcg_provider_uw_price
    ,CAST(NULL AS STRING) AS rcg_provider_acquired_by
    ,CAST(NULL AS DATE) AS rcg_provider_file_date
    ,CAST(NULL AS STRING) AS rcg_provider_eligibility
    ,CAST(NULL AS STRING) AS rcg_provider_initiator
    ,CAST(NULL AS STRING) AS rcg_provider_Servicer
    ,CAST(NULL AS STRING) AS rcg_provider_Auction_result_updated
    ,CAST(NULL AS STRING) AS rcg_provider_suspension_reason_grouped
    ,CAST(NULL AS STRING) AS rcg_provider_suspension_reason_full
    ,CAST(NULL AS BOOLEAN) AS rcg_provider_eligibility_flag_property
    ,CAST(NULL AS BOOLEAN) AS rcg_provider_eligible_for_reschedule
    ,CAST(NULL AS DECIMAL(5,2)) AS rcg_provider_marketability_score
    ,CAST(NULL AS INT) AS rcg_provider_auction_round

FROM  `{env}-cdh`.primary.auctionrealestate
WHERE BusinessDate = {run_date!r} and SourceSystem='qca'
'''
)

auctionrealestate_qca.createOrReplaceTempView("auctionrealestate_qca")

##### Final query with resulted DataFrame

In [0]:
final_df = spark.sql('''
    SELECT *
    FROM auctionrealestate_extfile
    UNION
    SELECT *
    FROM auctionrealestate_thesis
    UNION
    SELECT *
    FROM auctionrealestate_qca
''')


In [0]:
final_df.createOrReplaceTempView("final_df")

##### Write DataFrame to Delta Lake based on 'unity' value

In [0]:
if unity == "True":
    final_df.write \
        .format("delta") \
        .partitionBy("BusinessDate") \
        .mode("append") \
        .saveAsTable(derivative_table_path) # in order to write in delta_lake
else:
    final_df.write \
        .format("delta") \
        .partitionBy("BusinessDate") \
        .mode("append") \
        .save(derivative_table_path) # in order to write in delta_lake